## Configuration

In [0]:
project_identifier = 'dac005'

# IG thresholds: columns must have BOTH ig_risk <= max_ig_risk AND ig_severity <= max_ig_severity.
# Untagged columns are excluded unless listed in columns_to_include.
max_ig_risk = 3
max_ig_severity = 2

columns_to_exclude = ['ADC_UPDT']

# Manual includes for untagged columns you've reviewed.
# Format: {'catalog.schema.table': ['col1', 'col2']} or use '*' as table key for all tables.
columns_to_include = {}

# RDE tables (from 4_prod.rde) — same set as existing DAC005.
rde_tables = [
    'rde_all_procedures', 'rde_allergydetails', 'rde_apc_diagnosis', 'rde_apc_opcs',
    'rde_cds_apc', 'rde_cds_opa', 'rde_critactivity', 'rde_critopcs', 'rde_critperiod',
    'rde_emergencyd', 'rde_family_history', 'rde_measurements', 'rde_medadmin',
    'rde_op_diagnosis', 'rde_opa_opcs', 'rde_patient_demographics', 'rde_pc_diagnosis',
    'rde_pc_problems', 'rde_pc_procedures', 'rde_pharmacyorders', 'rde_pathology',
    'rde_raw_pathology', 'rde_all_diagnosis', 'rde_msds_booking', 'rde_emergency_findings',
]

# Bronze/map tables (from 4_prod.bronze).
map_tables = ['map_diagnosis', 'map_problem', 'map_procedure', 'map_med_admin']

# Bloodcount tables (from 4_prod.ancil_bloodcounts). SELECT *, joined via
# sample_no → rde_raw_pathology.LabNo → cohort.
bloodcount_tables = [
    'action_message', 'ana_internal1', 'ana_internal2', 'bb', 'bb_f', 'cal', 'cls_param',
    'dst_data', 'dst_data_2', 'err', 'flag_sus2', 'flagging', 'flg_gate', 'giveup', 'hsa',
    'hsa_f', 'ig', 'ipu_csv1', 'ipu_csv2', 'jo_aflas', 'jo_aflas01', 'jo_aflas03',
    'jo_aflas04', 'lyact', 'mix_issue', 'mt_data', 'neonew_hpc', 'neuteoissue', 'otherinfo',
    'outputdata', 'plt_abn_sus', 'plt_clumps_wnr_gate', 'plt_clumps_wnr_v17_18',
    'plt_clumps_wnr_v21', 'plt_dst_raw_data', 'plt_swt', 'rbc_abn_sus', 'rbc_dst_raw_data',
    'reportable', 'reportable_f', 'research', 'research_f', 'sampling_plt', 'sampling_pltf',
    'sampling_rbc', 'sampling_ret', 'sampling_wdf', 'sampling_wdf_2times', 'sampling_wnr',
    'sampling_wpc', 'sampling_wpc_2times', 'sampling_wpc_3times', 'sampling_wpc_4times',
    'service_in', 'service_out', 'servicesettinglog', 't_data', 'wbc_abn_sct_bf',
    'wbc_abn_sus', 'wdf_low_sfl', 'wnr_aggl', 'wp_reana', 'xn_sample', 'sct_ret',
    'sct_wdf', 'sct_wnr',
]

## Cohort — all patients with XN bloodcounts

In [0]:
cohort_sql = f"""
CREATE OR REPLACE VIEW 6_mgmt.cohorts.{project_identifier} AS
SELECT DISTINCT
    p.PERSON_ID,
    CAST(NULL AS STRING) AS SUBCOHORT
FROM 4_prod.rde.rde_raw_pathology p
INNER JOIN 4_prod.ancil_bloodcounts.xn_sample x
    ON p.LabNo = x.sample_no_
WHERE p.PERSON_ID IS NOT NULL
"""
spark.sql(cohort_sql)
print(f"Created cohort view: 6_mgmt.cohorts.{project_identifier}")

## Schema Setup

In [0]:
spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

existing_views_df = spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}")
existing_views_set = set()
if existing_views_df.count() > 0:
    existing_views_set = {row.viewName for row in existing_views_df.collect()}
    for view_name in existing_views_set:
        spark.sql(f"DROP VIEW IF EXISTS 5_projects.{project_identifier}.{view_name}")
        print(f"Dropped view: 5_projects.{project_identifier}.{view_name}")

existing_tables_df = spark.sql(f"SHOW TABLES IN 5_projects.{project_identifier}")
for row in existing_tables_df.collect():
    if row.tableName not in existing_views_set and row.tableName != project_identifier:
        spark.sql(f"DROP TABLE IF EXISTS 5_projects.{project_identifier}.{row.tableName}")
        print(f"Dropped table: 5_projects.{project_identifier}.{row.tableName}")

## Helper Functions

In [0]:
def get_safe_columns(catalog, schema, table):
    """
    Return columns passing IG filtering.

    A column is included only if BOTH ig_risk <= max_ig_risk AND ig_severity <= max_ig_severity,
    OR it appears in columns_to_include for this table. Untagged columns are EXCLUDED
    (treated as max risk). columns_to_exclude always wins.
    """
    full_path = f"{catalog}.{schema}.{table}"
    all_columns = spark.table(full_path).columns

    safe_df = spark.sql(f"""
        SELECT r.column_name
        FROM (
            SELECT column_name, CAST(tag_value AS INT) AS risk_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_risk'
        ) r
        JOIN (
            SELECT column_name, CAST(tag_value AS INT) AS sev_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_severity'
        ) s ON r.column_name = s.column_name
        WHERE r.risk_val <= {max_ig_risk}
          AND s.sev_val <= {max_ig_severity}
    """)
    safe_columns = set(safe_df.toPandas()['column_name'].tolist())

    includes = set(columns_to_include.get(full_path, []))
    includes |= set(columns_to_include.get('*', []))
    safe_columns |= includes
    safe_columns -= set(columns_to_exclude)

    return [c for c in all_columns if c in safe_columns]


def find_person_id_column(catalog, schema, table):
    """Find the person ID column in a table, checking common name variations."""
    full_path = f"{catalog}.{schema}.{table}"
    columns = spark.table(full_path).columns
    for candidate in ['PERSON_ID', 'person_id', 'Person_ID', 'PERSONID', 'PersonID']:
        if candidate in columns:
            return candidate
    for col in columns:
        if 'person' in col.lower() and 'id' in col.lower():
            return col
    return None


def find_sample_column(catalog, schema, table):
    """Find the analyser sample ID column in a bloodcount table."""
    full_path = f"{catalog}.{schema}.{table}"
    columns = spark.table(full_path).columns
    for candidate in ['sample_no_', 'sample_no', 'sampleid']:
        if candidate in columns:
            return candidate
    return None


def create_cohort_filtered_view(catalog, schema, table, project_id, column_list=None, alias=None):
    """Create a PERSON_ID-cohort-filtered view with IG column filtering."""
    full_path = f"{catalog}.{schema}.{table}"
    view_name = alias or table

    if column_list is None:
        column_list = get_safe_columns(catalog, schema, table)
        if not column_list:
            print(f"WARNING: No columns passed IG filter for {full_path}. Skipping.")
            return False

    person_id_col = find_person_id_column(catalog, schema, table)
    columns_sql = "t.*" if column_list == ['*'] else ", ".join([f"t.`{c}`" for c in column_list])

    if person_id_col:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        INNER JOIN 6_mgmt.cohorts.{project_id} c
            ON t.{person_id_col} = c.PERSON_ID
        """
    else:
        print(f"Note: No person ID column in {full_path}. Creating view without cohort filtering.")
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        """

    spark.sql(view_sql)
    ncols = 'all' if column_list == ['*'] else str(len(column_list))
    print(f"Created view: 5_projects.{project_id}.{view_name} ({ncols} columns)")
    return True


def create_bloodcount_view(table, project_id):
    """Create a bloodcount view: SELECT * joined via sample_no -> rde_raw_pathology.LabNo -> cohort."""
    full_path = f"4_prod.ancil_bloodcounts.{table}"
    sample_col = find_sample_column('4_prod', 'ancil_bloodcounts', table)
    if not sample_col:
        print(f"WARNING: No sample_no column in {full_path}. Skipping.")
        return False
    spark.sql(f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{table} AS
        SELECT b.*
        FROM {full_path} b
        INNER JOIN 4_prod.rde.rde_raw_pathology p
            ON b.`{sample_col}` = p.LabNo
        INNER JOIN 6_mgmt.cohorts.{project_id} c
            ON p.PERSON_ID = c.PERSON_ID
    """)
    print(f"Created view: 5_projects.{project_id}.{table} (bloodcounts, all columns)")
    return True

## Create Views

In [0]:
created = []
failed = []

# --- RDE tables ---
for table in rde_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'rde', table, project_identifier):
            created.append(f"rde.{table}")
    except Exception as e:
        failed.append((f"rde.{table}", str(e)))
        print(f"ERROR: rde.{table}: {e}")

# --- Bronze/map tables ---
for table in map_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'bronze', table, project_identifier):
            created.append(f"bronze.{table}")
    except Exception as e:
        failed.append((f"bronze.{table}", str(e)))
        print(f"ERROR: bronze.{table}: {e}")

# --- Bloodcount tables (sample_no bridge via rde_raw_pathology) ---
for table in bloodcount_tables:
    try:
        if create_bloodcount_view(table, project_identifier):
            created.append(f"ancil_bloodcounts.{table}")
    except Exception as e:
        failed.append((f"ancil_bloodcounts.{table}", str(e)))
        print(f"ERROR: ancil_bloodcounts.{table}: {e}")

## Schema Documentation View

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.schema AS
SELECT
    table_name,
    column_name,
    COALESCE(comment, '') AS column_comment
FROM 5_projects.information_schema.columns
WHERE table_catalog = '5_projects'
  AND table_schema = '{project_identifier}'
  AND table_name != 'schema'
ORDER BY table_name, column_name
""")
print(f"Created schema view: 5_projects.{project_identifier}.schema")

## Summary

In [0]:
print("=" * 60)
print(f"Project Setup Complete: {project_identifier}")
print("=" * 60)
print(f"\nCohort:  6_mgmt.cohorts.{project_identifier}")
print(f"Schema:  5_projects.{project_identifier}")
print(f"\nIG Thresholds: ig_risk <= {max_ig_risk}, ig_severity <= {max_ig_severity}")
print("Policy: untagged columns are EXCLUDED (treated as max risk).")
print("Bloodcount tables bypass IG filter (SELECT *, analyser outputs).")
print(f"\nViews created: {len(created)}")
for v in created:
    print(f"  + {v}")
if failed:
    print(f"\nFailed: {len(failed)}")
    for t, e in failed:
        print(f"  x {t}: {e}")